## Path setup and load everything

In [2]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))

import config
import models
import attacks
import evaluation

import numpy as np
import torch
import joblib

# Load the preprocessed arrays and fitted objects from Stage 2.
data = np.load(config.PROCESSED_DIR / "stage2_arrays.npz")
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]
label_encoder = joblib.load(config.PROCESSED_DIR / "label_encoder.joblib")
class_names = list(label_encoder.classes_)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loaded. X_test: {X_test.shape}  device: {device}")

Loaded. X_test: (718, 9)  device: cuda


## Retrain the baseline CNN

In [3]:
from sklearn.utils.class_weight import compute_class_weight

# Class weights as same as stage 3
w = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(w, dtype=torch.float32, device=device)

# Train the baseline CNN on the full duplicated training set
cnn = models.CNN1D(n_features=9, n_classes=6)
cnn = models.train_cnn(
    cnn, X_train, y_train,
    n_epochs=50, device=device,
    class_weights=class_weights,
    random_seed=config.RANDOM_SEED,
)

print("Baseline CNN trained.")

    epoch   1/50     loss 1.6285
    epoch   5/50     loss 0.1573
    epoch  10/50     loss 0.0149
    epoch  15/50     loss 0.0034
    epoch  20/50     loss 0.0021
    epoch  25/50     loss 0.0015
    epoch  30/50     loss 0.0012
    epoch  35/50     loss 0.0009
    epoch  40/50     loss 0.0009
    epoch  45/50     loss 0.0007
    epoch  50/50     loss 0.0007
Baseline CNN trained.


## Wrap CNN and generate FGSM examples

In [4]:
# Wrap the trained CNN
classifier = attacks.wrap_cnn_for_art(cnn, n_features=9, n_classes=6, device=device)

# Baseline: how does the CNN do on the clean test set
cnn.eval()
with torch.no_grad():
    clean_logits = cnn(torch.tensor(X_test, dtype=torch.float32, device=device))
    clean_pred = clean_logits.argmax(dim=1).cpu().numpy()

from sklearn.metrics import f1_score
clean_f1 = f1_score(y_test, clean_pred, average="macro", zero_division=0)
print(f"CNN clean macro-F1 (reference): {clean_f1:.4f}\n")

# Generate FGSM adversarial examples from the test set at a moderate epsilon
X_adv_fgsm = attacks.generate_fgsm(classifier, X_test, epsilon=0.10)

# How does the CNN do on the purturbated test set
with torch.no_grad():
    adv_logits = cnn(torch.tensor(X_adv_fgsm, dtype=torch.float32, device=device))
    adv_pred = adv_logits.argmax(dim=1).cpu().numpy()

adv_f1 = f1_score(y_test, adv_pred, average="macro", zero_division=0)
print(f"CNN FGSM macro-F1 (epsilon=0.10): {adv_f1:.4f}")
print(f"Degredation: {clean_f1 - adv_f1:.4f} drop")

CNN clean macro-F1 (reference): 0.6606

CNN FGSM macro-F1 (epsilon=0.10): 0.1432
Degredation: 0.5174 drop


## Epsilon sweep, FGSM and PGD, against both models

In [5]:
from sklearn.metrics import f1_score

# Helper: get macro-F1 for a model's predictions on some inputs
def cnn_macro_f1(X):
    cnn.eval()
    with torch.no_grad():
        pred = cnn(torch.tensor(X, dtype=torch.float32, device=device)).argmax(dim=1).cpu().numpy()
    return f1_score(y_test, pred, average="macro", zero_division=0)

def rf_macro_f1(X):
    return f1_score(y_test, rf.predict(X), average="macro", zero_division=0)

# Retrain the RF so the notebook is self-contained
rf = models.build_random_forest(random_seed=config.RANDOM_SEED)
rf.fit(X_train, y_train)

# Clean baselines
clean_cnn = cnn_macro_f1(X_test)
clean_rf = rf_macro_f1(X_test)
print(f"CLEAN   CNN={clean_cnn:.4f}     RF={clean_rf:.4f}\n")

# Sweep over the epsilon list from config for both attacks
epsilons = config.FGSM_EPSILONS

results = {"eps": [], "fgsm_cnn": [], "fgsm_rf": [], "pgd_cnn": [], "pgd_rf": []}

for eps in epsilons:
    # FGSM at this epsilon crafted on the cnn
    Xf = attacks.generate_fgsm(classifier, X_test, epsilon=eps)
    # PGD at this epsilon crafted on the cnn
    Xp = attacks.generate_pgd(classifier, X_test, epsilon=eps)

    # Feed same cradfted frames to both models (transfer attack)
    results["eps"].append(eps)
    results["fgsm_cnn"].append(cnn_macro_f1(Xf))
    results["fgsm_rf"].append(rf_macro_f1(Xf))
    results["pgd_cnn"].append(cnn_macro_f1(Xp))
    results["pgd_rf"].append(rf_macro_f1(Xp))

    print(f"eps={eps:.2f} | FGSM: CNN={results['fgsm_cnn'][-1]:.4f} RF={results['fgsm_rf'][-1]:.4f}"
          f" | PGD: CNN={results['pgd_cnn'][-1]:.4f} RF={results['pgd_rf'][-1]:.4f}")

CLEAN   CNN=0.6606     RF=0.7761



PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.01 | FGSM: CNN=0.6606 RF=0.4995 | PGD: CNN=0.6606 RF=0.4995


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.05 | FGSM: CNN=0.3921 RF=0.2214 | PGD: CNN=0.4467 RF=0.1658


...

eps=0.10 | FGSM: CNN=0.1432 RF=0.1657 | PGD: CNN=0.1494 RF=0.1658


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.20 | FGSM: CNN=0.1278 RF=0.1619 | PGD: CNN=0.0728 RF=0.1523


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

eps=0.30 | FGSM: CNN=0.1588 RF=0.1655 | PGD: CNN=0.0725 RF=0.1636


## Verify the perturbation and break down per-class

In [6]:
from sklearn.metrics import classification_report

# Part 1: is the perturbation real at eps=0.01
X_adv_001 = attacks.generate_fgsm(classifier, X_test, epsilon=0.01)

# How much did the inputs actually change?
diff = np.abs(X_adv_001 - X_test)
print("PERTURBATION CHECK at eps=0.01")
print(f"    max abs change per feature  : {diff.max():.5f}")
print(f"    mean abs change             : {diff.mean():.5f}")
print(f"    rows that changed at all    : {(diff.sum(axis=1) > 0).sum()} / {len(X_test)}")

# Did the CNN predictions actually change?
cnn.eval()
with torch.no_grad():
    p_clean = cnn(torch.tensor(X_test, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    p_adv = cnn(torch.tensor(X_adv_001, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
print(f"    CNN predictions changed     : {(p_clean != p_adv).sum()} / {len(X_test)}")

# Part 2: per-class breakdown, RF under FGSM eps=0.01
print("\nRF per-class CLEAN:")
print(classification_report(y_test, rf.predict(X_test), target_names=class_names, zero_division=0))
print("\nRF per-class under FGSM eps=0.01:")
print(classification_report(y_test, rf.predict(X_adv_001), target_names=class_names, zero_division=0))

PERTURBATION CHECK at eps=0.01
    max abs change per feature  : 0.01000
    mean abs change             : 0.00835
    rows that changed at all    : 718 / 718
    CNN predictions changed     : 0 / 718

RF per-class CLEAN:
                         precision    recall  f1-score   support

                    DoS       1.00      0.75      0.86         4
                 benign       1.00      1.00      1.00       709
           spoofing-GAS       0.00      0.00      0.00         1
           spoofing-RPM       0.67      1.00      0.80         2
         spoofing-SPEED       1.00      1.00      1.00         1
spoofing-STEERING_WHEEL       1.00      1.00      1.00         1

               accuracy                           1.00       718
              macro avg       0.78      0.79      0.78       718
           weighted avg       1.00      1.00      1.00       718


RF per-class under FGSM eps=0.01:
                         precision    recall  f1-score   support

                    DoS 

## Integer-valid realism check

In [7]:
# Load the fitted scaler to map [0,1] with real CAN byte space (0-255)
scaler = joblib.load(config.PROCESSED_DIR / "feature_scaler.joblib")

def round_to_valid_can(X_scaled):
    """
    Round adversarial frames to legal integer CAN bytes, then re-scale.

    Steps: inverse transform [0,1] -> original 0-255 space, round to nearest integer, clip to 0-255 and then re-scale back to [0-1] for the models.
    The result is a frame an attacker could actually transmit on the bus.
    """
    X_real = scaler.inverse_transform(X_scaled)
    X_real = np.clip(np.round(X_real), config.FEATURE_MIN, config.FEATURE_MAX)
    X_valid = scaler.transform(X_real)
    return X_valid


print("INTEGER-VALID REALISM CHECK (FGSM)\n")
print(f"{'eps':>6} | {'continuous':>22} | {'integer-valid':>22}")
print(f"{'':>6} | {'CNN':>10} {'RF':>10} | {'CNN':>10} {'RF':>10}")

for eps in config.FGSM_EPSILONS:
    # Continuous attack as before
    Xc = attacks.generate_fgsm(classifier, X_test, epsilon=eps)
    # Integer-valid version of the same attack
    Xv = round_to_valid_can(Xc)

    row = (cnn_macro_f1(Xc), rf_macro_f1(Xc), cnn_macro_f1(Xv), rf_macro_f1(Xv))
    print(f"{eps:>6.2f} | {row[0]:>10.4f} {row[1]:>10.4f} | {row[2]:>10.4f} {row[3]:>10.4f}")

INTEGER-VALID REALISM CHECK (FGSM)

   eps |             continuous |          integer-valid
       |        CNN         RF |        CNN         RF
  0.01 |     0.6606     0.4995 |     0.2064     0.1542
  0.05 |     0.3921     0.2214 |     0.1959     0.1668
  0.10 |     0.1432     0.1657 |     0.1012     0.1637
  0.20 |     0.1278     0.1619 |     0.1322     0.1648
  0.30 |     0.1588     0.1655 |     0.1383     0.1609


## CNN per-class breakdown under attack

In [8]:
# It is confirmed that the CNN is untouched at eps=0.01.
# the interesting CNN collapse happen at eps=0.05.
# Therefore, let's see which classes fail there, to mirror the RF per-class analysis

X_adv_05 = attacks.generate_fgsm(classifier, X_test, epsilon=0.05)

cnn.eval()
with torch.no_grad():
    cnn_pred_05 = cnn(torch.tensor(X_adv_05, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    
print("CNN per-class CLEAN:")
print(classification_report(y_test, clean_pred, target_names=class_names, zero_division=0))
print("\nCNN per-class under FGSM eps=0.05:")
print(classification_report(y_test, cnn_pred_05, target_names=class_names, zero_division=0))

CNN per-class CLEAN:
                         precision    recall  f1-score   support

                    DoS       0.67      1.00      0.80         4
                 benign       1.00      1.00      1.00       709
           spoofing-GAS       1.00      1.00      1.00         1
           spoofing-RPM       0.50      0.50      0.50         2
         spoofing-SPEED       0.50      1.00      0.67         1
spoofing-STEERING_WHEEL       0.00      0.00      0.00         1

               accuracy                           0.99       718
              macro avg       0.61      0.75      0.66       718
           weighted avg       0.99      0.99      0.99       718


CNN per-class under FGSM eps=0.05:
                         precision    recall  f1-score   support

                    DoS       0.75      0.75      0.75         4
                 benign       1.00      0.88      0.94       709
           spoofing-GAS       0.00      0.00      0.00         1
           spoofing-RPM      

## Robust-support metric and the dual-metric sweep

In [9]:
from sklearn.metrics import f1_score

ROBUST_LABELS = [0, 1, 3]       # DoS, benign, spoofing-RPM

def macro_f1_full(y_true, y_pred):
    """6-class macro-F1"""
    return f1_score(y_true, y_pred, average="macro", zero_division=0)

def macro_f1_robust(y_true, y_pred):
    """Macro-F1 over only the robust-support classes"""
    return f1_score(y_true, y_pred, labels=ROBUST_LABELS, average="macro", zero_division=0)

def cnn_pred(X):
    cnn.eval()
    with torch.no_grad():
        return cnn(torch.tensor(X, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    
# Regenerate the FGSM sweep reporting both metrics for both models
print("DUAL-METRIC FGSM SWEEP  (full 6-class | robust-support 3-class)\n")
print(f"{'eps':>6} | {'CNN full':>9} {'CNN rob':>8} | {'RF full':>9} {'RF rob':>8}")

clean_cp = cnn_pred(X_test); clean_rp = rf.predict(X_test)
print(f"{'clean':>6} | {macro_f1_full(y_test,clean_cp):>9.4f} {macro_f1_robust(y_test,clean_cp):>8.4f} "
      f"| {macro_f1_full(y_test,clean_rp):>9.4f} {macro_f1_robust(y_test,clean_rp):>8.4f}")

for eps in config.FGSM_EPSILONS:
    Xf = attacks.generate_fgsm(classifier, X_test, epsilon=eps)
    cp = cnn_pred(Xf); rp = rf.predict(Xf)
    print(f"{eps:>6.2f} | {macro_f1_full(y_test,cp):>9.4f} {macro_f1_robust(y_test,cp):>8.4f} "
          f"| {macro_f1_full(y_test,rp):>9.4f} {macro_f1_robust(y_test,rp):>8.4f}")

DUAL-METRIC FGSM SWEEP  (full 6-class | robust-support 3-class)

   eps |  CNN full  CNN rob |   RF full   RF rob
 clean |    0.6606   0.7657 |    0.7761   0.8855
  0.01 |    0.6606   0.7657 |    0.4995   0.6657
  0.05 |    0.3921   0.7841 |    0.2214   0.4428
  0.10 |    0.1432   0.2865 |    0.1657   0.3315
  0.20 |    0.1278   0.1444 |    0.1619   0.3238
  0.30 |    0.1588   0.1843 |    0.1655   0.3310


# Train both defended CNNs

In [10]:
import defense

# Defended model 1: PGD only adversarial training
print(">>> Training defended CNN (PGD-only)")
cnn_pgd = defense.adversarial_train_cnn(
    X_train, y_train, strategy="pgd",
    epsilons=config.FGSM_EPSILONS, n_epochs=50,
    device=device, class_weights=class_weights,
    random_seed=config.RANDOM_SEED,
)

# Defended model 2: multi-strategy (FGSM+PGD)
print("\n>>> Training defended CNN (multi-strategy)")
cnn_multi = defense.adversarial_train_cnn(
    X_train, y_train, strategy="multi",
    epsilons=config.FGSM_EPSILONS, n_epochs=50,
    device=device, class_weights=class_weights,
    random_seed=config.RANDOM_SEED,
)

print("\nBoth defended models trained.")

>>> Training defended CNN (PGD-only)
    epoch   1/50     loss 1.6152
    epoch   5/50     loss 0.1699
    epoch  10/50     loss 0.0097
    epoch  15/50     loss 0.0032
    epoch  20/50     loss 0.0023
    epoch  25/50     loss 0.0017
    epoch  30/50     loss 0.0013
    epoch  35/50     loss 0.0010
    epoch  40/50     loss 0.0010
    epoch  45/50     loss 0.0018
    epoch  50/50     loss 0.0010


PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

...

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/120 [00:00<?, ?it/s]

    augmented trainset: 3838 clean -> 42218 total (multi strategy)
    epoch   1/50     loss 0.5687
    epoch   5/50     loss 0.0024
    epoch  10/50     loss 0.0034
    epoch  15/50     loss 0.0004
    epoch  20/50     loss 0.0002
    epoch  25/50     loss 0.0002
    epoch  30/50     loss 0.0002
    epoch  35/50     loss 0.0001
    epoch  40/50     loss 0.0000
    epoch  45/50     loss 0.0000
    epoch  50/50     loss 0.0020

Both defended models trained.


## Full evaluation of all four models across the attack sweep

In [11]:
# Predict helper for any CNN model
def pred_of(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
    
# The models to compare
def eval_all_on(X):
    return {
        "base_cnn":     macro_f1_robust(y_test, pred_of(cnn, X)),
        "def_pgd":      macro_f1_robust(y_test, pred_of(cnn_pgd, X)),
        "def_multi":    macro_f1_robust(y_test, pred_of(cnn_multi, X)),
        "rf":           macro_f1_robust(y_test, rf.predict(X)),
    }

print("ROBUST-SUPPORT macro-F1 (benign, DoS, RPM) under PGD attack\n")
print(f"{'eps':>6} | {'base':>7} {'def_pgd':>7} {'def_multi':>9} {'rf':>7}")

# Clean row
c = eval_all_on(X_test)
print(f"{'clean':>6} | {c['base_cnn']:>7.4f} {c['def_pgd']:>7.4f} {c['def_multi']:>9.4f} {c['rf']:>7.4f}")

# PGD attack sweep (The defence's real test as PGD is the stronger attack.)
for eps in config.FGSM_EPSILONS:
    Xp = attacks.generate_pgd(classifier, X_test, epsilon=eps)
    r = eval_all_on(Xp)
    print(f"{eps:>6.2f} | {r['base_cnn']:>7.4f} {r['def_pgd']:>7.4f} {r['def_multi']:>9.4f} {r['rf']:>7.4f}")

ROBUST-SUPPORT macro-F1 (benign, DoS, RPM) under PGD attack

   eps |    base def_pgd def_multi      rf
 clean |  0.7657  0.8887    0.8516  0.8855


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.01 |  0.7657  0.8887    0.8884  0.6657


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.05 |  0.8933  0.7393    0.8884  0.3317


...

  0.10 |  0.2988  0.3215    0.5042  0.3317


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.20 |  0.1455  0.2853    0.6602  0.3046


PGD - Batches:   0%|          | 0/23 [00:00<?, ?it/s]

  0.30 |  0.1451  0.2824    0.6570  0.3272


## Defense tested against FGSM attacks

In [13]:
print("ROBUST-SUPPORT macro-F1 under FGSM attack (defence generalisation check)\n")
print(f"{'eps':>6} | {'base':>7} {'def_pgd':>7} {'def_multi':>9} {'rf':>7}")

# Clean row
c = eval_all_on(X_test)
print(f"{'clean':>6} | {c['base_cnn']:>7.4f} {c['def_pgd']:>7.4f} {c['def_multi']:>9.4f} {c['rf']:>7.4f}")

# FGSM attack sweep this time (the test attacks are FGSM, not PGD)
for eps in config.FGSM_EPSILONS:
    Xf = attacks.generate_fgsm(classifier, X_test, epsilon=eps)
    r = eval_all_on(Xf)
    print(f"{eps:>6.2f} | {r['base_cnn']:>7.4f} {r['def_pgd']:>7.4f} {r['def_multi']:>9.4f} {r['rf']:>7.4f}")

ROBUST-SUPPORT macro-F1 under FGSM attack (defence generalisation check)

   eps |    base def_pgd def_multi      rf
 clean |  0.8660  0.8519    0.8514  0.8855
  0.01 |  0.7618  0.8887    0.8514  0.7439
  0.05 |  0.4184  0.8882    0.8514  0.3218
  0.10 |  0.3193  0.6839    0.6345  0.3081
  0.20 |  0.1693  0.5494    0.4571  0.3211
  0.30 |  0.2157  0.3087    0.4397  0.3312


## Defended CV

In [12]:
import numpy as np
import pandas as pd
import crossval

strict = pd.read_csv(config.PROCESSED_DIR / "ciciov2024_strict.csv")

print("DEFENDED-MODEL CROSS-VALIDATION (PGD @ eps=0.10, 2-fold)\n")
cv_def = crossval.crossval_defended(
    strict,
    config.FEATURE_COLUMNS,
    attack_eps=0.10,
    n_splits=2,
    device=device,
    random_seed=config.RANDOM_SEED,
    cnn_epochs=50,
)

def ms(key):
    v = cv_def[key]
    return f"{np.mean(v):.4f} ± {np.std(v):.4f}"

print("\n" + "="*60)
print("ROBUST-SUPPORT macro-F1 (mean ± fold-spread)")
print("="*60)
print(f"{'model':>12} | {'clean':>16} | {'PGD eps=0.10':>16}")
print(f"{'base CNN':>12} | {ms('base_clean'):>16} | {ms('base_adv'):>16}")
print(f"{'def PGD':>12} | {ms('pgd_clean'):>16} | {ms('pgd_adv'):>16}")
print(f"{'def multi':>12} | {ms('multi_clean'):>16} | {ms('multi_adv'):>16}")
print(f"{'RF':>12} | {ms('rf_clean'):>16} | {ms('rf_adv'):>16}")

DEFENDED-MODEL CROSS-VALIDATION (PGD @ eps=0.10, 2-fold)

Label mapping:
    0 -> DoS
    1 -> benign
    2 -> spoofing-GAS
    3 -> spoofing-RPM
    4 -> spoofing-SPEED
    5 -> spoofing-STEERING_WHEEL
    epoch   1/50     loss 1.7342
    epoch   5/50     loss 0.3962
    epoch  10/50     loss 0.0218
    epoch  15/50     loss 0.0035
    epoch  20/50     loss 0.0017
    epoch  25/50     loss 0.0012
    epoch  30/50     loss 0.0009
    epoch  35/50     loss 0.0006
    epoch  40/50     loss 0.0005
    epoch  45/50     loss 0.0006
    epoch  50/50     loss 0.0004
    epoch   1/50     loss 1.7068
    epoch   5/50     loss 0.2201
    epoch  10/50     loss 0.0102
    epoch  15/50     loss 0.0032
    epoch  20/50     loss 0.0018
    epoch  25/50     loss 0.0012
    epoch  30/50     loss 0.0009
    epoch  35/50     loss 0.0005
    epoch  40/50     loss 0.0004
    epoch  45/50     loss 0.0005
    epoch  50/50     loss 0.0003


PGD - Batches:   0%|          | 0/87 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/87 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/87 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/87 [00:00<?, ?it/s]

...

PGD - Batches:   0%|          | 0/87 [00:00<?, ?it/s]

PGD - Batches:   0%|          | 0/87 [00:00<?, ?it/s]

    augmented trainset: 2774 clean -> 30514 total (multi strategy)
    epoch   1/50     loss 0.5895
    epoch   5/50     loss 0.0053
    epoch  10/50     loss 0.0025
    epoch  15/50     loss 0.0022
    epoch  20/50     loss 0.0011
    epoch  25/50     loss 0.0007
    epoch  30/50     loss 0.0003
    epoch  35/50     loss 0.0002
    epoch  40/50     loss 0.0002
    epoch  45/50     loss 0.0001
    epoch  50/50     loss 0.0003


PGD - Batches:   0%|          | 0/57 [00:00<?, ?it/s]

  fold 2: robust labels = [0, 1, 3] (names=['DoS', 'benign', 'spoofing-GAS', 'spoofing-RPM', 'spoofing-SPEED', 'spoofing-STEERING_WHEEL'])
  fold 2: base_adv=0.3389 pgd_adv=0.7090 multi_adv=0.6143 rf_adv=0.2830

ROBUST-SUPPORT macro-F1 (mean ± fold-spread)
       model |            clean |     PGD eps=0.10
    base CNN |  0.6460 ± 0.0200 |  0.3796 ± 0.0407
     def PGD |  0.7875 ± 0.0853 |  0.7543 ± 0.0453
   def multi |  0.6438 ± 0.0079 |  0.5495 ± 0.0648
          RF |  0.8528 ± 0.0039 |  0.2853 ± 0.0022


## Inference latency benchmark (CPU, single-frame)

In [13]:
import time
import numpy as np

# Move the CNN to CPU for an edge-realistic measurement
cnn_cpu = cnn.to("cpu")
cnn_cpu.eval()

# A single test frame, shaed as one sample
single = X_test[:1].astype(np.float32)

# --- CNN single-frame latency ---
import torch
single_t = torch.tensor(single, dtype=torch.float32)
# Warm up
with torch.no_grad():
    for _ in range(10):
        _ = cnn_cpu(single_t)

# Times runs
n_runs = 1000
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(n_runs):
        _ = cnn_cpu(single_t)
cnn_ms = (time.perf_counter() - t0) / n_runs * 1000

# --- RF single-frame latency ---
for _ in range(10):
    _ = rf.predict(single)
t0 = time.perf_counter()
for _ in range(n_runs):
    _ = rf.predict(single)
rf_ms = (time.perf_counter() - t0) / n_runs * 1000

print("INFERENCE LATENCY (CPU, single frame, mean of 1000 runs)")
print(f"  RF  : {rf_ms:.4f} ms/frame")
print(f"  CNN : {cnn_ms:.4f} ms/frame  (defended CNN identical - same architecture)")
print(f"\n  CAN frames can arrive ~1-2 ms apart on a busy bus; compare against that.")

# Restore CNN to original device for any further GPU work
cnn = cnn_cpu.to(device)

INFERENCE LATENCY (CPU, single frame, mean of 1000 runs)
  RF  : 7.0682 ms/frame
  CNN : 0.0572 ms/frame  (defended CNN identical - same architecture)

  CAN frames can arrive ~1-2 ms apart on a busy bus; compare against that.
